In [ ]:
# ============================================================================
# STAGE 8 -- CONSENSUS TRACE vs RANDOM MAJORITY-VOTE TRACE  (Qwen pairwise)
#
# STANDALONE single cell. Tests whether the ensemble's consensus reasoning
# trace is a BETTER justification than a plausible alternative.
#
# This version derives BOTH the consensus answer and the consensus trace
# DIRECTLY from the weighted ensemble DAG, applying the §3.5 rules:
#   1. consensus_answer = argmax_c W(c) over the answer-cluster root nodes
#      (with n_attest tiebreak), exactly as the paper specifies.
#   2. consensus trace  = recursive unfold from c*, picking
#                         S* = argmax_S phi(S) where phi(S) = min_{u in S} W(u)
#                         at every Or node.
# Stage 5's parsed JSONL is NO LONGER required -- we read only the Stage 4
# weighted ensemble DAG and the Stage 1 eval CSV.
#
# Per question, across all seed folders and both weighting schemes:
#   1. MAJORITY VOTE: majority of the 20 per-trace answers (Stage 1 CSV).
#   2. CONSENSUS ANSWER: argmax_c W(c) over the ensemble DAG's answer
#      clusters (read from the weighted JSONL, not parsed).
#   3. CONSENSUS TRACE: §3.5 unfold from the consensus answer's cluster.
#   4. RANDOM TRACE: random unfold rooted at the MAJORITY-VOTE answer's
#      cluster, length-matched to the consensus trace.
#   5. Linearize both subgraphs into reasoning chains.
#   6. Qwen3-32B picks which chain reasons better. Run TWICE with A/B
#      swapped to cancel position bias.
#   7. Report consensus-vs-random win rates per (scheme, dataset).
#
# Inputs (per seed folder):
#   <seed>/<ds>_eval_cots.csv                            (Stage 1)
#   <seed>/<ds>_ensemble_dags_weighted_<scheme>.jsonl    (Stage 4)
# Outputs (in OUT_DIR):
#   stage8_trace_comparison.csv
#   stage8_trace_summary.json
# ============================================================================

# ----------------------------------------------------------------------------
# CACHE REDIRECTION
# ----------------------------------------------------------------------------
import os

os.environ["VLLM_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_MOE_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_DEEP_GEMM_WARMUP"] = "skip"

HF_CACHE_ROOT = "/work/hdd/bfrc"
os.environ["HF_HOME"]                 = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_HUB_CACHE"]            = f"{HF_CACHE_ROOT}/hf/hub"
os.environ["TRANSFORMERS_CACHE"]      = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_DATASETS_CACHE"]       = f"{HF_CACHE_ROOT}/hf/datasets"
os.environ["VLLM_CACHE_ROOT"]         = f"{HF_CACHE_ROOT}/vllm"
os.environ["TRITON_CACHE_DIR"]        = f"{HF_CACHE_ROOT}/triton"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{HF_CACHE_ROOT}/torch_inductor"
os.environ["TMPDIR"]                  = f"{HF_CACHE_ROOT}/tmp"
for p in (os.environ["HF_HOME"], os.environ["HF_HUB_CACHE"],
          os.environ["HF_DATASETS_CACHE"], os.environ["VLLM_CACHE_ROOT"],
          os.environ["TRITON_CACHE_DIR"], os.environ["TORCHINDUCTOR_CACHE_DIR"],
          os.environ["TMPDIR"]):
    os.makedirs(p, exist_ok=True)

import importlib, sys
if "huggingface_hub" in sys.modules:
    importlib.reload(sys.modules["huggingface_hub"])
    if "huggingface_hub.constants" in sys.modules:
        importlib.reload(sys.modules["huggingface_hub.constants"])
from huggingface_hub import constants as _hf_constants
_resolved = str(_hf_constants.HF_HUB_CACHE)
print(f"[cache check] huggingface_hub resolved HF_HUB_CACHE = {_resolved}")
if not _resolved.startswith(HF_CACHE_ROOT):
    print(f"[cache check] WARNING: HF cache outside {HF_CACHE_ROOT}.")
else:
    print(f"[cache check] OK -- writing under {HF_CACHE_ROOT}")


# ----------------------------------------------------------------------------
# Imports
# ----------------------------------------------------------------------------
import csv, gc, glob, json, re, time, random
from collections import defaultdict, Counter
from typing import Dict, List, Optional, Set, Tuple, FrozenSet

import numpy as np
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

csv.field_size_limit(sys.maxsize)


# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SEED_DIRS = [
    "./outputs_2234",
    "./outputs_3234",
    "./outputs_4234",
    "./outputs_5234",
    "./outputs_1234",
]
SCHEME_TAGS      = ["simple", "accuracy"]
DATASETS_TO_EVAL = ["musr_ta", "musr_mm", "musr_op", "folio"]

OUT_DIR = SEED_DIRS[0]
RESULTS_CSV  = os.path.join(OUT_DIR, "stage8_trace_comparison.csv")
SUMMARY_JSON = os.path.join(OUT_DIR, "stage8_trace_summary.json")

SAMPLE_SEED    = 4242
N_RANDOM_TRIES = 200
NODE_MATCH_TOL = 1

JUDGE_MODEL    = "Qwen/Qwen3-32B"
ENABLE_THINKING = False
JUDGE_TEMP     = 0.0
JUDGE_TOP_P    = 1.0
JUDGE_MAX_TOK  = 256
JUDGE_MAX_LEN  = 16384
JUDGE_GPU_UTIL = 0.90
JUDGE_TP       = 1

VLLM_DOWNLOAD_DIR = f"{HF_CACHE_ROOT}/hf/hub"
os.makedirs(VLLM_DOWNLOAD_DIR, exist_ok=True)


# ============================================================================
# 1. MAJORITY VOTE
# ============================================================================
def _norm(s: str) -> str:
    return (s or "").strip().lower()


def majority_answers_for(seed_dir: str, dataset: str) -> Dict[str, str]:
    path = os.path.join(seed_dir, f"{dataset}_eval_cots.csv")
    if not os.path.exists(path):
        return {}
    votes: Dict[str, Counter] = defaultdict(Counter)
    with open(path, "r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            qid = (row.get("question_id") or "").strip()
            pred = (row.get("predicted_label") or "").strip()
            if qid and pred:
                votes[qid][pred] += 1
    majority: Dict[str, str] = {}
    for qid, c in votes.items():
        top = sorted(c.items(), key=lambda kv: (-kv[1], kv[0]))
        majority[qid] = top[0][0]
    return majority


# ============================================================================
# 2. LOAD STAGE 4 WEIGHTED ENSEMBLE DAGS
# ============================================================================
def load_weighted_index(seed_dir: str, dataset: str, scheme: str
                        ) -> Dict[str, Dict]:
    path = os.path.join(seed_dir,
                        f"{dataset}_ensemble_dags_weighted_{scheme}.jsonl")
    idx: Dict[str, Dict] = {}
    if not os.path.exists(path):
        return idx
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            qid = r.get("question_id")
            if qid is not None:
                idx[qid] = r
    return idx


# ============================================================================
# 3. CONSENSUS ANSWER FROM THE ENSEMBLE DAG (paper §3.5)
#
# consensus_answer = argmax_c W(c) over the answer-cluster ROOT NODES, with
# n_attest as tiebreak (same rule used by Stage 4.1's weighted consensus and
# Stage 5's parsed consensus, but computed locally so Stage 5 is not needed).
#
# Returns (answer_text, answer_cluster_id, all_clusters_sorted) where
# all_clusters_sorted is the full ranked list of candidates -- used so the
# majority-vote answer can also be mapped to a cluster id.
# ============================================================================
def derive_consensus(wrec: Dict, id_to_node: Dict[str, Dict],
                     W: Dict[str, float]
                     ) -> Tuple[Optional[str], Optional[str], List[Dict]]:
    answer_clusters = wrec.get("answer_clusters", []) or []
    # only keep clusters whose root node exists in the DAG
    valid = []
    for ac in answer_clusters:
        cid = ac.get("answer_cluster_id")
        if cid is None or cid not in id_to_node:
            continue
        valid.append({
            "text": ac.get("text"),
            "answer_cluster_id": cid,
            "root_W": float(W.get(cid, 0.0)),
            "n_attest": int(ac.get("n_attest", 0)),
        })
    if not valid:
        return None, None, []
    valid.sort(key=lambda p: (-p["root_W"], -p["n_attest"]))
    winner = valid[0]
    return winner["text"], winner["answer_cluster_id"], valid


def find_cluster_id_for_answer(all_clusters: List[Dict],
                               want_answer: str) -> Optional[str]:
    """Find the cluster id whose text matches want_answer (normalized).
    Used to locate the majority-vote answer's cluster in the ensemble."""
    want = _norm(want_answer)
    for ac in all_clusters:
        if _norm(ac.get("text")) == want:
            return ac.get("answer_cluster_id")
    return None


# ============================================================================
# 4. TRACE UNFOLDING from the weighted ensemble DAG
#
#   - 'weighted' : at each Or node take the support bundle of greatest
#                  phi(S) = min_{u in S} W(u)  -- the §3.5 consensus trace.
#   - 'random'   : at each Or node take a support bundle uniformly at random.
# ============================================================================
def phi_of_bundle(bundle: Dict, W: Dict[str, float]) -> float:
    members = bundle.get("members", [])
    if not members:
        return 0.0
    return min(W.get(m, 0.0) for m in members)


def unfold_trace(target: str, id_to_node: Dict[str, Dict],
                 W: Dict[str, float], mode: str,
                 rng: Optional[random.Random]) -> Dict:
    nodes: Set[str] = set()
    edges: List[Tuple[str, str]] = []

    def unfold(nid: str, on_path: FrozenSet[str]) -> None:
        if nid in on_path:
            nodes.add(nid)
            return
        node = id_to_node.get(nid)
        if node is None:
            nodes.add(nid)
            return
        nodes.add(nid)
        gate = node.get("gate", "Atomic")
        support = node.get("support", [])
        if gate == "Atomic" or not support:
            return
        if gate == "And":
            chosen = support[0]
        elif mode == "weighted":
            chosen = max(support, key=lambda S: phi_of_bundle(S, W))
        else:  # random
            chosen = support[rng.randrange(len(support))]
        new_path = on_path | {nid}
        for m in chosen.get("members", []):
            edges.append((nid, m))
            unfold(m, new_path)

    unfold(target, frozenset())
    return {"nodes": sorted(nodes), "edges": [[c, p] for (c, p) in edges]}


def attach_details(graph: Dict, id_to_node: Dict[str, Dict],
                    W: Dict[str, float]) -> Dict:
    graph["node_details"] = [{
        "id":   nid,
        "type": id_to_node.get(nid, {}).get("type"),
        "text": id_to_node.get(nid, {}).get("text"),
        "gate": id_to_node.get(nid, {}).get("gate"),
    } for nid in graph["nodes"]]
    graph["score"] = round(sum(W.get(x, 0.0) for x in graph["nodes"]), 6)
    return graph


# ============================================================================
# 5. LINEARIZE
# ============================================================================
def linearize_chain(graph: Dict) -> str:
    details = {d["id"]: d for d in (graph.get("node_details") or [])}
    node_ids = list(graph.get("nodes", []))
    idset = set(node_ids)

    preds: Dict[str, Set[str]] = {nid: set() for nid in node_ids}
    for edge in graph.get("edges", []):
        if len(edge) != 2:
            continue
        child, parent = edge
        if child in idset and parent in idset:
            preds[child].add(parent)

    indeg = {nid: len(preds[nid]) for nid in node_ids}
    children: Dict[str, List[str]] = defaultdict(list)
    for child, ps in preds.items():
        for p in ps:
            children[p].append(child)
    queue = [nid for nid in node_ids if indeg[nid] == 0]
    order: List[str] = []
    qi = 0
    while qi < len(queue):
        u = queue[qi]; qi += 1
        order.append(u)
        for c in children[u]:
            indeg[c] -= 1
            if indeg[c] == 0:
                queue.append(c)
    if len(order) < len(node_ids):
        seen = set(order)
        order += [nid for nid in node_ids if nid not in seen]

    lines = []
    for i, nid in enumerate(order, start=1):
        d = details.get(nid, {})
        lines.append(f"Step {i} [{d.get('type') or '?'}]: "
                     f"{(d.get('text') or '').strip()}")
    return "\n".join(lines)


# ============================================================================
# 6. JUDGE PROMPT
# ============================================================================
JUDGE_SYSTEM = (
    "You are a strict evaluator of reasoning quality. You are given two "
    "candidate reasoning chains, each arguing for a final answer to the same "
    "question. Decide which chain is the better reasoning and justification: "
    "better-grounded steps, sounder inferences, fewer gaps or irrelevant "
    "leaps. "
    "Chain LENGTH IS IRRELEVANT: a longer chain with more steps is NOT "
    "automatically better, and a shorter chain is NOT automatically worse. "
    "Do not reward verbosity. Judge only the soundness and grounding of the "
    "reasoning, not how many steps it has. "
    "Answer with a single token: A or B."
)


def build_prompt(question_blob: str,
                 chain_a: str, answer_a: str,
                 chain_b: str, answer_b: str) -> str:
    return f"""Two candidate reasoning chains each argue for an answer to the same
question. Decide which chain is the better reasoning and justification.

QUESTION CONTEXT:
{question_blob}

--- CHAIN A (concludes: {answer_a}) ---
{chain_a}

--- CHAIN B (concludes: {answer_b}) ---
{chain_b}

Which chain reasons better -- better-grounded steps, sounder inferences,
fewer gaps or irrelevant leaps? The number of steps does NOT matter; do not
prefer a chain merely because it is longer. Reply with exactly one token:
A or B."""


def strip_reasoning(raw: str) -> str:
    s = re.sub(r"<think>.*?</think>", " ", raw, flags=re.DOTALL | re.IGNORECASE)
    if "</think>" in s.lower():
        s = s[s.lower().rfind("</think>") + len("</think>"):]
    if "<think>" in s.lower():
        s = s[:s.lower().find("<think>")]
    return s.strip()


def extract_verdict(raw: str) -> Optional[str]:
    cleaned = strip_reasoning(raw)
    if not cleaned:
        return None
    up = cleaned.upper()
    tokens = re.findall(r"\b([AB])\b", up)
    if tokens:
        return tokens[-1]
    m = re.match(r"\s*([AB])[\).:,]", up)
    if m:
        return m.group(1)
    if up in ("A", "B"):
        return up
    return None


def apply_template(tok, messages) -> str:
    try:
        return tok.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False,
            enable_thinking=ENABLE_THINKING)
    except TypeError:
        return tok.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False)


# ============================================================================
# 7. BUILD WORK LIST
# ============================================================================
print("\n[stage 8] building comparisons across seeds ...")
rng = random.Random(SAMPLE_SEED)

work_items: List[Dict] = []
n_no_clusters = 0           # ensemble has no answer clusters at all
n_no_majtrace = 0           # majority answer not represented as a cluster
n_random_eq_consensus = 0
n_no_lengthmatch = 0

for seed_idx, seed_dir in enumerate(SEED_DIRS):
    if not os.path.isdir(seed_dir):
        print(f"  [seed {seed_idx}] MISSING dir, skipping: {seed_dir}")
        continue
    for dataset in DATASETS_TO_EVAL:
        majority = majority_answers_for(seed_dir, dataset)
        if not majority:
            print(f"  [seed {seed_idx}/{dataset}] no eval CSV, skipping")
            continue
        for scheme in SCHEME_TAGS:
            weighted_idx = load_weighted_index(seed_dir, dataset, scheme)
            if not weighted_idx:
                print(f"  [seed {seed_idx}/{scheme}/{dataset}] "
                      f"no weighted DAG file, skipping")
                continue

            n_made = 0
            for qid, maj_answer in majority.items():
                wrec = weighted_idx.get(qid)
                if wrec is None:
                    continue
                ensemble = wrec.get("ensemble_dag") or {}
                nodes = ensemble.get("nodes", [])
                id_to_node = {n["id"]: n for n in nodes}
                W = {n["id"]: float(n.get("W", 0.0)) for n in nodes}

                # ---- derive consensus answer + cluster id from the DAG ----
                cons_answer, cons_cid, all_clusters = derive_consensus(
                    wrec, id_to_node, W)
                if cons_answer is None or cons_cid is None:
                    n_no_clusters += 1
                    continue

                # ---- locate the majority-vote answer's cluster ------------
                maj_cid = find_cluster_id_for_answer(all_clusters, maj_answer)
                if maj_cid is None or maj_cid not in id_to_node:
                    n_no_majtrace += 1
                    continue

                # ---- consensus trace (§3.5 unfold) ------------------------
                cons_graph = attach_details(
                    unfold_trace(cons_cid, id_to_node, W, "weighted", None),
                    id_to_node, W)

                # ---- random trace, length-matched to consensus -----------
                cons_nodeset = frozenset(cons_graph["nodes"])
                cons_n = len(cons_graph["nodes"])
                best_cand = None
                best_diff = None
                for _ in range(N_RANDOM_TRIES):
                    cand = unfold_trace(maj_cid, id_to_node, W, "random", rng)
                    if frozenset(cand["nodes"]) == cons_nodeset:
                        continue
                    diff = abs(len(cand["nodes"]) - cons_n)
                    if best_diff is None or diff < best_diff:
                        best_diff = diff
                        best_cand = cand
                        if diff == 0:
                            break
                if best_cand is None:
                    n_random_eq_consensus += 1
                    continue
                if best_diff > NODE_MATCH_TOL:
                    n_no_lengthmatch += 1
                    continue
                rand_graph = attach_details(best_cand, id_to_node, W)

                cons_chain = linearize_chain(cons_graph)
                rand_chain = linearize_chain(rand_graph)

                work_items.append({
                    "seed_idx": seed_idx, "scheme": scheme,
                    "dataset": dataset, "question_id": qid,
                    "majority_answer":  maj_answer,
                    "consensus_answer": cons_answer,
                    "answers_match":    _norm(maj_answer) == _norm(cons_answer),
                    "gold_label":       wrec.get("gold_label"),
                    "cons_chain":  cons_chain,
                    "rand_chain":  rand_chain,
                    "cons_n_nodes": len(cons_graph["nodes"]),
                    "rand_n_nodes": len(rand_graph["nodes"]),
                    "cons_score":  cons_graph["score"],
                    "rand_score":  rand_graph["score"],
                    "votes_consensus": 0,
                    "votes_random":    0,
                    "n_judgements":    0,
                })
                n_made += 1

            print(f"  [seed {seed_idx}/{scheme}/{dataset}] "
                  f"{n_made} comparisons built")

print(f"\n[stage 8] {len(work_items)} comparisons built")
print(f"  skipped: no answer clusters={n_no_clusters}, "
      f"no majority-answer trace={n_no_majtrace}, "
      f"random==consensus only trace={n_random_eq_consensus}, "
      f"no length-matched random trace (>{NODE_MATCH_TOL} nodes)="
      f"{n_no_lengthmatch}")

# ---- length-balance sanity check ---------------------------------------
if work_items:
    cn = [w["cons_n_nodes"] for w in work_items]
    rn = [w["rand_n_nodes"] for w in work_items]
    diffs = [abs(a - b) for a, b in zip(cn, rn)]
    mc = sum(cn) / len(cn)
    mr = sum(rn) / len(rn)
    print(f"[stage 8] length-balance check after matching:")
    print(f"  mean nodes  consensus={mc:.2f}  random={mr:.2f}  "
          f"diff={mc-mr:+.2f}  (should be near 0)")
    print(f"  per-pair |node diff|  max={max(diffs)}  "
          f"(must be <= NODE_MATCH_TOL={NODE_MATCH_TOL})")
    over = sum(1 for d in diffs if d > NODE_MATCH_TOL)
    if over:
        print(f"  >> WARNING: {over} pairs exceed the tolerance -- "
              f"matching logic bug.")
    else:
        print(f"  >> OK: every pair is within the node-count tolerance.")


# ============================================================================
# 8. BUILD JUDGE PROMPTS
# ============================================================================
if not work_items:
    print("[stage 8] nothing to compare.")
else:
    print(f"\n[stage 8] loading tokenizer {JUDGE_MODEL}")
    tokenizer = AutoTokenizer.from_pretrained(
        JUDGE_MODEL, cache_dir=VLLM_DOWNLOAD_DIR, trust_remote_code=True)
    prompt_budget = JUDGE_MAX_LEN - JUDGE_MAX_TOK - 32

    flat_prompts: List[str] = []
    flat_meta: List[Dict] = []
    n_dropped_toolong = 0
    dropped_examples: List[str] = []

    for w_idx, w in enumerate(work_items):
        qblob = f"[dataset={w['dataset']}, question_id={w['question_id']}]"
        orders = [
            ("A", w["cons_chain"], w["consensus_answer"],
                  w["rand_chain"], w["majority_answer"]),
            ("B", w["rand_chain"], w["majority_answer"],
                  w["cons_chain"], w["consensus_answer"]),
        ]
        built = []
        fits = True
        for (cons_slot, cha, ansa, chb, ansb) in orders:
            user = build_prompt(qblob, cha, ansa, chb, ansb)
            messages = [{"role": "system", "content": JUDGE_SYSTEM},
                        {"role": "user",   "content": user}]
            rendered = apply_template(tokenizer, messages)
            if (w["cons_chain"] not in rendered) or \
               (w["rand_chain"] not in rendered):
                fits = False
                break
            ntok = len(tokenizer.encode(rendered, add_special_tokens=False))
            if ntok > prompt_budget:
                fits = False
                break
            built.append((cons_slot, rendered))
        if not fits:
            n_dropped_toolong += 1
            if len(dropped_examples) < 10:
                dropped_examples.append(
                    f"{w['seed_idx']}|{w['scheme']}|{w['dataset']}|"
                    f"{w['question_id']}")
            continue
        for (cons_slot, rendered) in built:
            flat_prompts.append(rendered)
            flat_meta.append({"work_idx": w_idx, "consensus_slot": cons_slot})

    print(f"[stage 8] {len(flat_prompts)} judge prompts queued")
    if n_dropped_toolong:
        print(f"[stage 8] DROPPED {n_dropped_toolong} comparisons "
              f"(prompt > {prompt_budget} tokens) -- NOT truncated.")
        print(f"          examples: {dropped_examples}")

    # ========================================================================
    # 9. JUDGE
    # ========================================================================
    print(f"\n[stage 8] loading judge {JUDGE_MODEL}")
    llm = LLM(
        model=JUDGE_MODEL,
        dtype="bfloat16",
        trust_remote_code=True,
        gpu_memory_utilization=JUDGE_GPU_UTIL,
        max_model_len=JUDGE_MAX_LEN,
        tensor_parallel_size=JUDGE_TP,
        download_dir=VLLM_DOWNLOAD_DIR,
    )
    sps = [SamplingParams(temperature=JUDGE_TEMP, top_p=JUDGE_TOP_P,
                          max_tokens=JUDGE_MAX_TOK, seed=0, n=1)
           for _ in flat_prompts]

    print(f"[stage 8] dispatching {len(flat_prompts)} prompts ...")
    t0 = time.time()
    outputs = llm.generate(flat_prompts, sps)
    dt = time.time() - t0
    print(f"[stage 8] judge done in {dt:.1f}s "
          f"({dt/max(1,len(flat_prompts)):.3f}s/call avg)")

    n_unparsed = 0
    for meta, out in zip(flat_meta, outputs):
        picked = extract_verdict(out.outputs[0].text)
        w = work_items[meta["work_idx"]]
        cons_slot = meta["consensus_slot"]
        w["n_judgements"] += 1
        if picked is None:
            n_unparsed += 1
            continue
        if picked == cons_slot:
            w["votes_consensus"] += 1
        else:
            w["votes_random"] += 1
    if n_unparsed:
        pct = 100.0 * n_unparsed / max(1, len(flat_meta))
        print(f"[stage 8] note: {n_unparsed}/{len(flat_meta)} judge outputs "
              f"({pct:.1f}%) unparseable -- skipped")

    del llm, outputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ========================================================================
    # 10. PER-QUESTION OUTCOME + CSV
    # ========================================================================
    results = []
    for w in work_items:
        if w["n_judgements"] == 0:
            continue
        vc, vr = w["votes_consensus"], w["votes_random"]
        if vc > vr:
            outcome = "consensus"
        elif vr > vc:
            outcome = "random"
        else:
            outcome = "tie"
        results.append({**w, "outcome": outcome})

    RES_FIELDS = ["seed_idx", "scheme", "dataset", "question_id",
                  "majority_answer", "consensus_answer", "answers_match",
                  "gold_label", "cons_n_nodes", "rand_n_nodes",
                  "cons_score", "rand_score",
                  "votes_consensus", "votes_random", "n_judgements",
                  "outcome", "cons_chain", "rand_chain"]
    print(f"\n[stage 8] writing per-question CSV: {RESULTS_CSV}")
    with open(RESULTS_CSV, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=RES_FIELDS, quoting=csv.QUOTE_ALL)
        wr.writeheader()
        for r in results:
            wr.writerow({k: r.get(k, "") for k in RES_FIELDS})

    # ========================================================================
    # 11. WIN RATES
    # ========================================================================
    def winrates(rows: List[Dict]) -> Dict:
        n_q = len(rows)
        votes_c = sum(r["votes_consensus"] for r in rows)
        votes_r = sum(r["votes_random"]    for r in rows)
        votes_t = votes_c + votes_r
        n_cons = sum(1 for r in rows if r["outcome"] == "consensus")
        n_rand = sum(1 for r in rows if r["outcome"] == "random")
        n_tie  = sum(1 for r in rows if r["outcome"] == "tie")
        decided = n_cons + n_rand
        return {
            "n_questions": n_q,
            "judgement_winrate_consensus":
                (votes_c / votes_t) if votes_t else float("nan"),
            "question_winrate_consensus":
                (n_cons / decided) if decided else float("nan"),
            "n_outcome_consensus": n_cons,
            "n_outcome_random":    n_rand,
            "n_outcome_tie":       n_tie,
        }

    agg: Dict[Tuple[str, str], List[Dict]] = defaultdict(list)
    for r in results:
        agg[(r["scheme"], r["dataset"])].append(r)

    print(f"\n{'='*78}")
    print("[stage 8] RESULTS -- how often Qwen prefers the CONSENSUS trace")
    print(f"{'='*78}")

    summary_rows = []
    for (scheme, dataset) in sorted(agg.keys()):
        rows = agg[(scheme, dataset)]
        overall = winrates(rows)
        matched = winrates([r for r in rows if r["answers_match"]])

        by_seed: Dict[int, List[Dict]] = defaultdict(list)
        for r in rows:
            by_seed[r["seed_idx"]].append(r)
        seed_q_wr = []
        for sidx, srows in by_seed.items():
            wr = winrates(srows)["question_winrate_consensus"]
            if wr == wr:
                seed_q_wr.append(wr)
        m_wr = float(np.mean(seed_q_wr)) if seed_q_wr else float("nan")
        s_wr = float(np.std(seed_q_wr))  if seed_q_wr else float("nan")

        print(f"\n  scheme={scheme:8s} dataset={dataset:6s}  "
              f"({overall['n_questions']} comparisons)")
        print(f"    per-judgement consensus win rate : "
              f"{100*overall['judgement_winrate_consensus']:.1f}%")
        print(f"    per-question  consensus win rate : "
              f"{100*overall['question_winrate_consensus']:.1f}%  "
              f"(consensus={overall['n_outcome_consensus']}, "
              f"random={overall['n_outcome_random']}, "
              f"tie={overall['n_outcome_tie']})")
        print(f"    per-question win rate, per-seed  : "
              f"{100*m_wr:.1f}% +/- {100*s_wr:.1f}%")
        if matched["n_questions"]:
            print(f"    [answers-match subset, {matched['n_questions']} q] "
                  f"per-question consensus win rate : "
                  f"{100*matched['question_winrate_consensus']:.1f}%")

        summary_rows.append({
            "scheme": scheme, "dataset": dataset,
            "overall": overall,
            "answers_match_subset": matched,
            "per_question_winrate_seed_mean": m_wr,
            "per_question_winrate_seed_std":  s_wr,
        })

    with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump({
            "config": {
                "judge_model": JUDGE_MODEL,
                "enable_thinking": ENABLE_THINKING,
                "judge_max_tok": JUDGE_MAX_TOK,
                "n_ab_orders": 2,
                "judge_max_len": JUDGE_MAX_LEN,
                "node_match_tol": NODE_MATCH_TOL,
                "comparisons_dropped_too_long": n_dropped_toolong,
                "skipped_no_clusters": n_no_clusters,
                "skipped_no_majority_trace": n_no_majtrace,
                "skipped_no_length_match": n_no_lengthmatch,
                "judge_outputs_unparseable": n_unparsed,
                "consensus_source": "derived directly from weighted DAG (§3.5)",
            },
            "results": summary_rows,
        }, f, indent=2)

    print(f"\n[stage 8] per-question CSV : {RESULTS_CSV}")
    print(f"[stage 8] summary JSON     : {SUMMARY_JSON}")